In [3]:
import numpy as np

def KNN(X_train, X_test, y_train, k):
    num_test = X_test.shape[0]
    num_train = X_train.shape[0]
    y_pred = np.zeros((num_test, num_train))

    # Tính ma trận khoảng cách giữa các điểm Test và dữ liệu Train
    for i in range(num_test):
        for j in range(num_train):
            y_pred[i, j] = np.sqrt(np.sum(np.power(X_test[i, :] - X_train[j, :], 2)))

    results = []
    # Sắp xếp khoảng cách tăng dần và lấy k láng giềng gần nhất
    for i in range(len(y_pred)):
        zipped = zip(y_pred[i, :], y_train)
        res = sorted(zipped, key=lambda x: x[0])
        results_topk = res[:k]

        # Đếm số lượng phiếu bầu của mỗi lớp bằng từ điển (Hòa phiếu lấy lớp xuất hiện trước)
        classes = {}
        for _, j in results_topk:
            j = int(j)
            if j not in classes:
                classes[j] = 1
            else:
                classes[j] = classes[j] + 1

        results.append(max(classes, key=classes.get))
    return np.array(results)

# Cách 1: Đánh giá thủ công bằng trực quan hình ảnh / Độ chính xác tập Test
def evaluate_k_manually(X_train, X_test, y_train, y_test, k_values=[1, 3, 5]):
    print("-> Cách 1: Đánh giá thủ công các giá trị k:")
    for k in k_values:
        y_pred = KNN(X_train, X_test, y_train, k)
        accuracy = np.mean(y_pred == y_test) * 100
        print(f"   Hệ số k = {k} mang lại độ chính xác tập test: {accuracy:.2f}%")

# Cách 2: Đánh giá tự động qua K-Fold Cross Validation (Mô phỏng GridSearchCV thủ công)
def evaluate_k_cross_validation(X_train, y_train, max_k=5, cv_folds=3):
    print("-> Cách 2: Tự động đánh giá k qua Cross-Validation (K-Fold):")
    best_k = 1
    best_avg_acc = -1.0

    indices = np.arange(X_train.shape[0])
    folds = np.array_split(indices, cv_folds)

    for k in range(1, max_k + 1):
        fold_accuracies = []
        for i in range(cv_folds):
            test_idx = folds[i]
            train_idx = np.hstack([folds[j] for j in range(cv_folds) if j != i])

            val_pred = KNN(X_train[train_idx], X_train[test_idx], y_train[train_idx], k)
            acc = np.mean(val_pred == y_train[test_idx])
            fold_accuracies.append(acc)

        avg_acc = np.mean(fold_accuracies)
        print(f"   k = {k} | Độ chính xác trung bình trên các tập con: {avg_acc * 100:.2f}%")
        if avg_acc > best_avg_acc:
            best_avg_acc = avg_acc
            best_k = k

    print(f"   ==> Tự động chọn được k tối ưu nhất là: k = {best_k}")
    return best_k

# --- KHỐI THỬ NGHIỆM BÀI 2 ---
if __name__ == "__main__":
    print("=== THỬ NGHIỆM BÀI 2: K-NN VỚI ĐÁNH GIÁ TRỌNG SỐ K ===")
    X_tr = np.array([[2, 0], [2, 2], [3, 0], [2, 3], [4, 0], [4, 2]])
    y_tr = np.array([0, 1, 0, 1, 1, 0]) # Nhãn lớp 0 và lớp 1
    X_te = np.array([[0, 0], [3, 3]])
    y_te = np.array([0, 1]) # Nhãn thực tế tập test để đánh giá mô hình

    evaluate_k_manually(X_tr, X_te, y_tr, y_te, k_values=[1, 3])
    evaluate_k_cross_validation(X_tr, y_tr, max_k=4, cv_folds=3)
    print("-" * 50)

=== THỬ NGHIỆM BÀI 2: K-NN VỚI ĐÁNH GIÁ TRỌNG SỐ K ===
-> Cách 1: Đánh giá thủ công các giá trị k:
   Hệ số k = 1 mang lại độ chính xác tập test: 100.00%
   Hệ số k = 3 mang lại độ chính xác tập test: 100.00%
-> Cách 2: Tự động đánh giá k qua Cross-Validation (K-Fold):
   k = 1 | Độ chính xác trung bình trên các tập con: 66.67%
   k = 2 | Độ chính xác trung bình trên các tập con: 66.67%
   k = 3 | Độ chính xác trung bình trên các tập con: 16.67%
   k = 4 | Độ chính xác trung bình trên các tập con: 66.67%
   ==> Tự động chọn được k tối ưu nhất là: k = 1
--------------------------------------------------
